# Week 7 · Gradient Boosting in Practice — In-Class Workbook

**AI & Machine Learning for Business · MSc in Business Analytics and AI**

This workbook is the hands-on companion to the Week 7 lecture. It is deliberately separate from the **Save or Let Go** methodology presentation and the **BookingGuard** take-home exercise.

### How to work

1. Write a prediction or hypothesis before each experiment.
2. Change only the cells marked **YOU CHANGE**.
3. Record the result and a keep/reject/follow-up decision.
4. Do not use the held-out test period until the final section.

**Recommended environment:** Google Colab. If XGBoost or LightGBM is unavailable, uncomment the install line in the setup cell.


In [ ]:
# Colab: uncomment if needed
# !pip -q install xgboost lightgbm

import time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, average_precision_score, roc_auc_score, brier_score_loss, mean_squared_error
from sklearn.calibration import calibration_curve
import xgboost as xgb, lightgbm as lgb, sklearn

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("sklearn", sklearn.__version__, "| xgboost", xgb.__version__, "| lightgbm", lgb.__version__)

## §1 · Intuition: boosting as a team of editors

**The analogy for class.** Think of a first draft of a forecast written by an intern who says "50 for everyone."
A first editor reads it and fixes only the biggest, broadest mistake ("weekends are higher").
A second editor reads the *edited* draft and fixes what is still wrong ("rainy Friday nights are lower").
Each editor is cautious: they only apply **a fraction** of their suggested change (the learning rate).
You stop hiring editors when the draft stops improving **on documents they have never seen** (validation).

**Real-world anchors to use on the slide:**
- **Demand forecasting (QSR / retail):** a base weekly average, then trees that correct for daypart, weather, promo overlap.
- **Card fraud:** most transactions are easy; later trees concentrate on the hard, rare patterns earlier trees got wrong.
- **Telecom churn:** a base churn rate, then corrections for contract end, network complaints, bill shock × tenure.
- **Search / feed ranking:** boosted trees (e.g. LambdaMART) were the workhorse ranker in web search and ads for years.

Below, we watch one "editor" at a time on a 1-D toy problem.

In [ ]:
# Toy data: a noisy S-curve
n = 60
x = np.sort(rng.uniform(0, 10, n))
y = 2.2 * np.tanh((x - 5) / 2.5) + rng.normal(0, 0.2, n)
X1 = x.reshape(-1, 1)

eta, F = 0.5, np.full(n, y.mean())          # start: predict the mean for everyone
grid = np.linspace(0, 10, 400).reshape(-1, 1)
F_grid = np.full(400, y.mean())

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for m, ax in enumerate(axes):
    ax.scatter(x, y, s=12, color="#1f5aa6")
    ax.plot(grid, F_grid, color="#e07b00", lw=2)
    resid = y - F
    ax.set_title(f"Round {m}: train MSE = {np.mean(resid**2):.3f}", fontsize=10)
    tree = DecisionTreeRegressor(max_depth=1).fit(X1, resid)   # one 'editor' = a stump
    F = F + eta * tree.predict(X1)
    F_grid = F_grid + eta * tree.predict(grid)
plt.suptitle("Each stump fixes part of what the current ensemble still gets wrong", y=1.03)
plt.tight_layout(); plt.show()

**Discussion prompt (60 seconds):** Why does round 1 split near *x = 5*, while later rounds split elsewhere?
*(Answer: the biggest remaining error moves. Each tree sees a different target — the residual — not the original y.)*

## §2 · From pseudocode to working code

The slide pseudocode:

```
initialize F₀(x) from the training target
for m in 1..M:
    compute the negative gradient of the loss
    fit a shallow tree hₘ(x) to that signal
    update Fₘ(x) = Fₘ₋₁(x) + η·hₘ(x)
    record training and validation loss
    retain the best validated iteration
return F_best(x)
```

Every line maps to one line of Python. Read the function side by side with the pseudocode.

In [ ]:
def boost_regression(X_tr, y_tr, X_va, y_va, M=200, eta=0.1, depth=2):
    F0 = y_tr.mean()                                   # initialize F0
    F_tr, F_va = np.full(len(y_tr), F0), np.full(len(y_va), F0)
    trees, hist = [], []
    for m in range(M):                                 # for m in 1..M
        neg_grad = y_tr - F_tr                         #   negative gradient of squared error = residual
        h = DecisionTreeRegressor(max_depth=depth, random_state=SEED).fit(X_tr, neg_grad)  # fit shallow tree
        F_tr += eta * h.predict(X_tr)                  #   update F_m = F_{m-1} + eta * h_m
        F_va += eta * h.predict(X_va)
        trees.append(h)
        hist.append((mean_squared_error(y_tr, F_tr), mean_squared_error(y_va, F_va)))  # record losses
    hist = np.array(hist)
    best = int(hist[:, 1].argmin()) + 1                # retain best validated iteration
    return dict(F0=F0, trees=trees[:best], eta=eta, best=best, hist=hist)

def predict(model, X):
    return model["F0"] + model["eta"] * sum(t.predict(X) for t in model["trees"])

In [ ]:
# Sanity check: our 15-line booster vs scikit-learn's GradientBoostingRegressor
Xr = rng.uniform(0, 10, (1500, 3))
yr = np.sin(Xr[:, 0]) + 0.5 * Xr[:, 1] * (Xr[:, 2] > 5) + rng.normal(0, 0.3, 1500)
Xtr, Xva, ytr, yva = Xr[:1000], Xr[1000:], yr[:1000], yr[1000:]

ours = boost_regression(Xtr, ytr, Xva, yva, M=300, eta=0.1, depth=2)
skl = GradientBoostingRegressor(n_estimators=ours["best"], learning_rate=0.1, max_depth=2,
                                random_state=SEED).fit(Xtr, ytr)
print(f"Best round (ours): {ours['best']}")
print(f"Validation MSE  ours: {mean_squared_error(yva, predict(ours, Xva)):.4f}   "
      f"sklearn: {mean_squared_error(yva, skl.predict(Xva)):.4f}")

The two numbers should be nearly identical. That is the teaching moment: **the library is our loop plus engineering.**
XGBoost and LightGBM add regularization, histograms, second-order information and speed — not a new idea.

## §3 · Lab check: the residual table (slide 32)

Fill the final column by hand first, then run the cell to check.

In [ ]:
lab = pd.DataFrame({"row": list("ABC"), "y": [1, 0, 1], "F_current": [0.40, 0.30, 0.75], "tree_h": [0.50, -0.20, 0.20]})
lab["residual"] = lab["y"] - lab["F_current"]
for eta in (0.1, 0.5, 1.0):
    lab[f"F_new (eta={eta})"] = (lab["F_current"] + eta * lab["tree_h"]).round(3)
lab

**Interpretation:** η scales the *size* of each step; the sign of `tree_h` (learned from the residual) sets the *direction*.
At η = 1.0 row A jumps to 0.90 in one step — fast, but every tree's mistakes are applied in full.

In [ ]:
# The classification version (slide 25): log-loss gradient on the score (log-odds) scale
sigmoid = lambda z: 1 / (1 + np.exp(-z))
p = 0.35; z = np.log(p / (1 - p))
print(f"y=1, current p={p}  -> negative gradient (y - p) = {1 - p:+.2f}")
print(f"leaf value +0.80, eta=0.10 -> score moves {z:.3f} -> {z + 0.08:.3f}, p moves {p:.3f} -> {sigmoid(z + 0.08):.3f}")

## Experiment contract

For every parameter experiment, complete this record **before** and **after** running the code.

| Run | One change | Prediction before run | Validation result | Keep / reject / follow-up | Why? |
|---|---|---|---|---|---|
| LR-1 | learning rate |  |  |  |  |
| D-1 | tree depth |  |  |  |  |
| B0–B3 | implementation parameters |  |  |  |  |

The goal is not to find the lowest number by trial-and-error. The goal is to make a defensible comparison.


## §4 · Learning rate × rounds: one capacity decision

**Real-world framing:** a small learning rate is like a cautious pricing team that moves prices 1% at a time —
slower to converge, but less likely to overreact to one noisy week.

In [ ]:
# YOU CHANGE — learning-rate experiment
# Predict first: which setting will need the most rounds, and which is most likely to overfit early?
LEARNING_RATES = [0.30, 0.10, 0.03]
MAX_ROUNDS = 600

print({'learning_rates': LEARNING_RATES, 'max_rounds': MAX_ROUNDS})


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for eta, c in zip(LEARNING_RATES, ["#c0392b", "#e07b00", "#1f5aa6"]):
    r = boost_regression(Xtr, ytr, Xva, yva, M=MAX_ROUNDS, eta=eta, depth=3)
    ax[0].plot(r["hist"][:, 0], color=c, label=f"eta={eta}")
    ax[1].plot(r["hist"][:, 1], color=c, label=f"eta={eta} (best round {r['best']})")
    ax[1].scatter(r["best"] - 1, r["hist"][r["best"] - 1, 1], color=c, zorder=3)
ax[0].set_title("Training MSE keeps falling"); ax[1].set_title("Validation MSE has a best round")
for a in ax: a.set_xlabel("boosting round"); a.legend(frameon=False, fontsize=8)
ax[1].set_ylim(0.08, 0.3)
plt.tight_layout(); plt.show()

**Reading the chart:** halving η roughly doubles the rounds needed. Training loss never tells you when to stop —
only validation does. This is slide 30's answer (B: restore the best round) in picture form.

## §5 · Tree depth: how complex is each correction?

Depth 1 can only add main effects. Our target contains an interaction (`x1 × [x2 > 5]`), so depth 1 should underfit.

In [ ]:
# YOU CHANGE — correction-tree capacity
# Predict first: which depth will underfit? Which may become unstable?
DEPTHS = [1, 2, 3, 6]

print({'depths_to_compare': DEPTHS})


In [ ]:
rows = []
for d in DEPTHS:
    r = boost_regression(Xtr, ytr, Xva, yva, M=500, eta=0.1, depth=d)
    rows.append({"depth": d, "best_round": r["best"], "train_MSE_at_best": r["hist"][r["best"]-1, 0].round(4),
                 "val_MSE_at_best": r["hist"][r["best"]-1, 1].round(4)})
pd.DataFrame(rows)

## §6 · A business dataset: subscription churn with a time split

Synthetic streaming-subscription data, 20,000 customers over 24 monthly cohorts.
**Prediction moment:** 30 days before renewal. **Action:** offer a retention discount.

The generator deliberately includes:
- non-linear effects (a bill-shock step, an engagement cliff), an interaction (new × monthly), and a rare high-risk segment;
- **drift**: later cohorts churn more (a competitor launched), so a random split would look optimistic;
- one **leaky** field, `cancel_reason_code`, populated only after churn. We will drop it — then show what happens if we don't.

In [ ]:
def make_churn(n=20000, seed=SEED):
    g = np.random.default_rng(seed)
    cohort = g.integers(0, 24, n)
    tenure = g.gamma(2.0, 8.0, n).clip(1, 72)
    contract = g.choice(["monthly", "annual", "bundle"], n, p=[0.55, 0.30, 0.15])
    region = g.choice(["Northeast", "South", "Midwest", "West"], n)
    bill_change = g.normal(0, 8, n)                     # % change in bill vs last quarter
    support_tickets = g.poisson(1.0, n)
    hours_watched = g.gamma(3, 6, n)
    devices = g.integers(1, 6, n)
    z = (-2.2 + 0.04 * cohort
         + 1.6 * (contract == "monthly") * np.exp(-tenure / 6)  # new monthly customers are fragile
         - 0.6 * (contract == "bundle")
         + 2.0 * (bill_change > 10)                              # bill shock is a step, not a slope
         + 0.5 * np.minimum(support_tickets, 4)
         - 0.9 * (hours_watched > 15)                            # engaged viewers stay
         + 2.0 * ((support_tickets >= 3) & (hours_watched < 8))  # rare high-risk segment
         - 0.15 * devices)
    churn = g.binomial(1, 1 / (1 + np.exp(-z)))
    leak = np.where(churn == 1, g.integers(1, 6, n), 0)      # only known AFTER churn
    return pd.DataFrame(dict(cohort=cohort, tenure=tenure, contract=contract, region=region,
                             bill_change=bill_change, support_tickets=support_tickets,
                             hours_watched=hours_watched, devices=devices,
                             cancel_reason_code=leak, churned=churn))

df = make_churn()
print(df.shape); print(df.groupby(pd.cut(df.cohort, [-1, 15, 19, 23]))["churned"].agg(["size", "mean"]).round(3))

In [ ]:
# Forward (time) split: train on cohorts 0-15, early-stop/validate on 16-19, test on 20-23 (hidden until the end)
target, LEAKY = "churned", ["cancel_reason_code"]
num = ["tenure", "bill_change", "support_tickets", "hours_watched", "devices"]
cat = ["contract", "region"]
train, valid, test = df[df.cohort <= 15], df[df.cohort.between(16, 19)], df[df.cohort >= 20]

def to_X(d):   # tree libraries: categories as integer codes; same columns for every model
    X = d[num + cat].copy()
    for c in cat: X[c] = X[c].astype("category")
    return X

Xtr, Xva, Xte = to_X(train), to_X(valid), to_X(test)
ytr, yva, yte = train[target].values, valid[target].values, test[target].values
print({k: len(v) for k, v in dict(train=train, valid=valid, test=test).items()})

## §7 · Fair comparison: same rows, same split, same metrics

This is slide 38 as code. Only the model and its declared configuration change.

### Metric primer: what is the Brier score?

For a binary outcome, the **Brier score** is the mean squared error of the predicted probabilities:

\[\text{Brier}=\frac{1}{n}\sum_{i=1}^{n}(p_i-y_i)^2\]

- **Lower is better; 0 is perfect.**
- It rewards probabilities that are close to the observed 0/1 outcomes.
- Unlike AUC, it is sensitive to probability calibration.
- Compare it on the same validation rows and outcome definition.

Use **log loss** as the primary probability-quality metric, **PR-AUC** for ranking under class imbalance, and **Brier** as a calibration-sensitive check.


### Implementation intuition before the benchmark

All three boosting libraries build additive tree corrections. They differ mainly in engineering and regularization choices.

| Method | Plain-English intuition | Typical real-world fit |
|---|---|---|
| Histogram Gradient Boosting | Strong, accessible baseline in scikit-learn | Medium-sized tabular risk or retention models |
| XGBoost | Regularized boosted trees with mature sparse-data and production tooling | Credit risk, fraud, search/ranking, sparse business tables |
| LightGBM | Histogram bins plus leaf-wise growth for speed on large tables | Demand forecasting, advertising, telemetry, very large tabular data |

These are starting hypotheses, not automatic choices. Use the same rows, split, metrics, and business constraints to select the final model.


In [ ]:
def evaluate(name, p, t0, best_round="n/a", config=""):
    return {"model": name, "config": config, "log_loss": log_loss(yva, p), "PR_AUC": average_precision_score(yva, p),
            "ROC_AUC": roc_auc_score(yva, p), "Brier": brier_score_loss(yva, p),
            "runtime_s": time.time() - t0, "best_round": best_round}

results, preds = [], {}
ohe_pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

# 1. Logistic baseline
t0 = time.time()
logit = Pipeline([("pre", ohe_pre), ("clf", LogisticRegression(max_iter=1000))]).fit(train, ytr)
preds["Logistic"] = logit.predict_proba(valid)[:, 1]
results.append(evaluate("Logistic", preds["Logistic"], t0, config="scaled + one-hot"))

# 2. Random Forest (Week 6 benchmark)
t0 = time.time()
rf = Pipeline([("pre", ohe_pre), ("clf", RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=SEED))]).fit(train, ytr)
preds["Random Forest"] = rf.predict_proba(valid)[:, 1]
results.append(evaluate("Random Forest", preds["Random Forest"], t0, config="300 trees, leaf 20"))

# 3. XGBoost (slide 44 config)
t0 = time.time()
xgb_m = xgb.XGBClassifier(n_estimators=800, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                          early_stopping_rounds=50, eval_metric="logloss", enable_categorical=True,
                          tree_method="hist", random_state=SEED)
xgb_m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
preds["XGBoost"] = xgb_m.predict_proba(Xva)[:, 1]
results.append(evaluate("XGBoost", preds["XGBoost"], t0, xgb_m.best_iteration + 1, "eta .05, depth 3, sample .8"))

# 4. LightGBM (slide 44 config)
t0 = time.time()
lgb_m = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.05, num_leaves=31, min_child_samples=30,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
lgb_m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
preds["LightGBM"] = lgb_m.predict_proba(Xva)[:, 1]
results.append(evaluate("LightGBM", preds["LightGBM"], t0, lgb_m.best_iteration_, "eta .05, 31 leaves, min leaf 30"))

# 5. scikit-learn HistGB (local benchmark)
t0 = time.time()
hgb = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=800, max_depth=3, categorical_features="from_dtype",
                                     early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=SEED).fit(Xtr, ytr)
preds["HistGB"] = hgb.predict_proba(Xva)[:, 1]
results.append(evaluate("HistGB", preds["HistGB"], t0, hgb.n_iter_, "eta .05, depth 3, internal ES"))

comp = pd.DataFrame(results).set_index("model").round(4)
comp

**Discussion:** Boosting clearly beats logistic here (PR-AUC ≈ 0.51 vs 0.43) because the churn drivers are steps and
interactions a linear score cannot express. Its edge over Random Forest is much smaller — is *that* gap material, or noise? Note too that HistGB used a *random internal* validation slice for early stopping —
that breaks the time contract. Ask the room to spot it before you point it out.

## §8 · Parameter laboratory: change one thing, record what happened

Complete the hypothesis column before fitting. Change one parameter family at a time. After each run, write a keep/reject/follow-up decision grounded in later-period validation evidence.


In [ ]:
# YOU CHANGE — define the runs before fitting
PARAMETER_RUNS = [
    # run, learning_rate, num_leaves, min_child_samples, max_rounds, hypothesis
    ('B0', 0.05, 15, 30, 400, 'Reference run'),
    ('B1', 0.02, 15, 30, 900, 'Smaller steps may improve validation but need more rounds'),
    ('B2', 0.05, 31, 30, 400, 'More leaves may capture interactions but raise variance'),
    ('B3', 0.05, 15, 80, 400, 'Larger leaves may regularize noisy segments'),
]

pd.DataFrame(PARAMETER_RUNS, columns=['run','learning_rate','num_leaves','min_child_samples','max_rounds','hypothesis'])


In [ ]:
grid = PARAMETER_RUNS
lab_rows = []
for run, lr, leaves, min_leaf, rounds, q in grid:
    t0 = time.time()
    m = lgb.LGBMClassifier(n_estimators=rounds, learning_rate=lr, num_leaves=leaves, min_child_samples=min_leaf,
                           random_state=SEED, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict_proba(Xva)[:, 1]
    lab_rows.append({"run": run, "lr": lr, "leaves": leaves, "min_leaf": min_leaf, "best_round": m.best_iteration_,
                     "val_log_loss": round(log_loss(yva, p), 4), "PR_AUC": round(average_precision_score(yva, p), 4),
                     "runtime_s": round(time.time() - t0, 2), "question": q})
pd.DataFrame(lab_rows)

experiment_log = pd.DataFrame(lab_rows)
experiment_log['decision'] = ''       # YOU WRITE: keep / reject / follow-up
experiment_log['evidence'] = ''       # YOU WRITE: cite validation evidence
display(experiment_log)


**Write one decision sentence per run** (keep / reject / follow-up). Note how B1 needs more rounds to reach a similar loss.

## §9 · From probabilities to a decision

A better log loss is not yet a business case. Three checks before anyone sends a retention offer:
calibration, threshold economics under a capacity limit, and segment errors.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.2))
for name, c in [("Logistic", "#888"), ("Random Forest", "#e07b00"), ("XGBoost", "#1f5aa6")]:
    fr, mp = calibration_curve(yva, preds[name], n_bins=10, strategy="quantile")
    ax.plot(mp, fr, marker="o", ms=4, color=c, label=name)
ax.plot([0, 1], [0, 1], ls="--", color="#bbb")
ax.set_xlabel("predicted churn probability"); ax.set_ylabel("observed churn rate")
ax.set_title("Calibration on the validation period"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# Threshold economics: offer costs $15 (discount + handling); a saved churner is worth $250; offer saves 30% of true churners.
# Capacity: the retention team can make at most 600 offers in the validation window.
COST_OFFER, VALUE_SAVED, SAVE_RATE, CAPACITY = 15, 250, 0.30, 600
p = preds["XGBoost"]
rows = []
for t in np.arange(0.10, 0.65, 0.05):
    flag = p >= t
    tp = int((flag & (yva == 1)).sum())
    ev = tp * SAVE_RATE * VALUE_SAVED - flag.sum() * COST_OFFER
    rows.append({"threshold": round(t, 2), "offers": int(flag.sum()), "true_churners_reached": tp,
                 "expected_value_$": int(ev), "feasible": flag.sum() <= CAPACITY})
policy = pd.DataFrame(rows)
best = policy[policy.feasible].sort_values("expected_value_$", ascending=False).iloc[0]
print(f"Best feasible threshold: {best.threshold}  |  offers: {best.offers}  |  EV: ${best['expected_value_$']:,}")
policy

In [ ]:
# Segment check at the chosen threshold: who bears the false positives?
t = best.threshold
seg = valid.assign(p=preds["XGBoost"], flag=preds["XGBoost"] >= t)
def seg_stats(g):
    return pd.Series({"n": len(g), "churn_rate": g.churned.mean(), "offer_rate": g.flag.mean(),
                      "recall": g.loc[g.churned == 1, "flag"].mean(),
                      "false_pos_share": (g.flag & (g.churned == 0)).sum() / max(g.flag.sum(), 1),
                      "mean_pred_minus_obs": g.p.mean() - g.churned.mean()})
seg.groupby("contract").apply(seg_stats).round(3)

**What the room should notice:** the value-maximizing threshold (0.20) needs ~760 offers — infeasible.
Capacity, not the model, sets the policy. Every segment's mean prediction sits *below* its observed churn:
the later cohorts drift upward, so a model trained on the past under-predicts. That is a recalibration
conversation, not a model-family one.

## §10 · Leakage demo: why flexible models make it worse

Add back the post-outcome field and watch the "improvement."

In [ ]:
X_leak_tr = Xtr.assign(cancel_reason_code=train.cancel_reason_code.values)
X_leak_va = Xva.assign(cancel_reason_code=valid.cancel_reason_code.values)
leaky = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, enable_categorical=True,
                          tree_method="hist", random_state=SEED).fit(X_leak_tr, ytr)
print(f"Honest XGBoost ROC-AUC : {roc_auc_score(yva, preds['XGBoost']):.3f}")
print(f"Leaky  XGBoost ROC-AUC : {roc_auc_score(yva, leaky.predict_proba(X_leak_va)[:, 1]):.3f}   <- too good to be true")

A near-perfect AUC is a **symptom**, not a success. At the prediction moment (30 days before renewal),
`cancel_reason_code` does not exist yet. Always ask: *could I have known this value at the moment I must act?*

## §11 · Final test — once

Only after the model, configuration and threshold are frozen.

In [ ]:
p_te = xgb_m.predict_proba(Xte)[:, 1]
print(f"Test (cohorts 20-23)  log loss {log_loss(yte, p_te):.4f} | PR-AUC {average_precision_score(yte, p_te):.4f} | "
      f"offers at t={best.threshold}: {(p_te >= best.threshold).sum()}")

## Required experiment record and recommendation

Submit the completed table below with your notebook.

| Run | Configuration | Hypothesis | Validation evidence | Decision | Risk / uncertainty |
|---|---|---|---|---|---|
| Baseline |  |  |  |  |  |
| B0 |  |  |  |  |  |
| B1 |  |  |  |  |  |
| B2 |  |  |  |  |  |
| B3 |  |  |  |  |  |

**Four-line recommendation**

1. Selected model and exact configuration.
2. Two pieces of later-period validation evidence.
3. One failure mode, uncertainty, or segment concern.
4. One next experiment or operational safeguard.


Test log loss is worse than validation: drift continued into cohorts 20–23. Report it honestly; do not re-tune.

## Exit ticket
1. Which configuration would you carry forward, and what two pieces of evidence support it?
2. Which segment worries you at the chosen threshold, and why?
3. What is the single next experiment, stated as a hypothesis?